# CO5085 E2 — Transformer: MSA và Tokenization

Notebook này được xây dựng để chạy thí nghiệm E2 trên Google Colab theo cùng kiểu quy trình như E1: kiểm tra môi trường, cài đặt phụ thuộc, smoke test, train, kiểm tra output và so sánh kết quả thực tế.

## 1. Cài đặt thư viện và kiểm tra môi trường Colab

Tôi bắt đầu bằng việc xác nhận bản PyTorch, trạng thái GPU và môi trường Colab trước khi đi vào các phần huấn luyện thực tế.

In [ ]:
!nvidia-smi

import torch

print("PyTorch version:", torch.__version__)
print("GPU is available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU Device Name:", torch.cuda.get_device_name(0))

In [ ]:
import os

if not os.path.exists("/content/deep-learning-computer-vision-co5085"):
    %cd /content
    !git clone https://github.com/hoctran19-cyber/deep-learning-computer-vision-co5085.git
    %cd /content/deep-learning-computer-vision-co5085/E2

!pwd
!ls -la

In [ ]:
!cat configs/default.yaml

!find . -maxdepth 3 -type f | sort

## 2. Cài đặt phụ thuộc và import module của dự án

Sau khi xác nhận môi trường, tôi cài đặt các gói cần thiết và import các module chính của E2 để chuẩn bị cho các kiểm tra tiếp theo.

In [ ]:
!pip install -r requirements.txt

In [ ]:
import os
import sys
from pathlib import Path

repo_path = "/content/deep-learning-computer-vision-co5085/E2"
if os.path.exists(repo_path) and repo_path not in sys.path:
    sys.path.insert(0, repo_path)

from src.utils import get_device, load_config, set_seed
from src.data import load_datasets, create_dataloaders
from src.model import build_model
from src.tokenizers import PatchTokenizer, RowTokenizer, CNNStemTokenizer
from src.attention import CustomMultiHeadSelfAttention, TorchMultiHeadSelfAttention

print("Imports OK!")

## 3. Tải dataset và kiểm tra batch đầu vào

Trước khi huấn luyện, tôi xác nhận dữ liệu được tải đúng và tensor có shape phù hợp với mô hình Transformer.

In [ ]:
config = load_config("configs/default.yaml")
set_seed(config["seed"])
bundle = load_datasets(config, Path.cwd())
loaders = create_dataloaders(bundle, config)
print(bundle.input_shape, bundle.num_classes, bundle.class_names)
images, labels = next(iter(loaders["train"]))
print("Batch:", images.shape, labels.shape)

## 4. Smoke test tokenizer

Mỗi tokenizer tạo token theo cách khác nhau; tôi kiểm tra số lượng token và shape đầu ra để đảm bảo tương thích với mô hình.

In [ ]:
embedding_dim = config["model"]["embedding_dim"]
tokenizers = {
    "patch": PatchTokenizer(bundle.input_shape, config["model"]["patch_size"], embedding_dim),
    "row": RowTokenizer(bundle.input_shape, embedding_dim),
    "cnn_stem": CNNStemTokenizer(bundle.input_shape, embedding_dim),
}
for name, tokenizer in tokenizers.items():
    tokens = tokenizer(images[:4])
    print(name, "num_tokens =", tokenizer.num_tokens, "shape =", tuple(tokens.shape))

## 5. Smoke test Multi-Head Self-Attention

Tôi kiểm tra cả phiên bản custom MSA và PyTorch MSA để đảm bảo gradient và output shape đều ổn định.

In [ ]:
x = torch.randn(2, 8, embedding_dim, requires_grad=True)
for name, module in {
    "custom": CustomMultiHeadSelfAttention(embedding_dim, config["model"]["num_heads"]),
    "torch": TorchMultiHeadSelfAttention(embedding_dim, config["model"]["num_heads"]),
}.items():
    y = module(x)
    y.mean().backward()
    print(name, tuple(y.shape), "grad ok:", x.grad is not None)

## 6. Smoke test mô hình hoàn chỉnh

Trước khi chạy các thí nghiệm dài, tôi thực hiện một bước forward/backward ngắn để xác nhận mô hình hoạt động đúng.

In [ ]:
model = build_model(config, bundle.input_shape, bundle.num_classes).to(get_device())
batch = images[:8].to(get_device())
targets = labels[:8].to(get_device())
logits = model(batch)
loss = torch.nn.CrossEntropyLoss()(logits, targets)
loss.backward()
print("logits:", logits.shape, "loss:", float(loss.detach().cpu()))

## 7. Kiểm tra CLI training

Tôi xác nhận script huấn luyện có thể chạy theo CLI trước khi thực hiện các thí nghiệm lớn hơn.

In [ ]:
!python -m src.train --help

In [ ]:
!python -m src.train --config configs/default.yaml --experiment-name smoke_custom_patch --attention custom --tokenizer patch --epochs 1 --batch-size 64

## 8. Huấn luyện thí nghiệm A: so sánh MSA

Bộ thí nghiệm A giữ nguyên tokenizer patch, chỉ thay đổi custom MSA và PyTorch MSA để so sánh hiệu suất và độ ổn định.

In [ ]:
!python -m src.train --config configs/default.yaml --experiment-name msa_custom_patch --attention custom --tokenizer patch
!python -m src.train --config configs/default.yaml --experiment-name msa_torch_patch --attention torch --tokenizer patch

## 9. Huấn luyện thí nghiệm B: so sánh tokenizer

Bộ thí nghiệm B giữ nguyên custom MSA, so sánh hiệu quả của patch, row và cnn_stem tokenizer.

In [ ]:
!python -m src.train --config configs/default.yaml --experiment-name tok_patch_custom --attention custom --tokenizer patch
!python -m src.train --config configs/default.yaml --experiment-name tok_row_custom --attention custom --tokenizer row
!python -m src.train --config configs/default.yaml --experiment-name tok_cnn_stem_custom --attention custom --tokenizer cnn_stem

## 10. Kiểm tra kết quả đầu ra

Sau khi chạy xong các thí nghiệm, tôi kiểm tra các file metrics, CSV so sánh và hình ảnh learning curves / confusion matrix được sinh ra.

In [ ]:
!find outputs -maxdepth 3 -type f | sort

import pandas as pd
from pathlib import Path

comparison_path = Path("outputs/metrics/comparison.csv")
if comparison_path.exists():
    display(pd.read_csv(comparison_path))
else:
    print("Chưa có comparison.csv vì thí nghiệm chưa chạy xong.")

In [ ]:
from IPython.display import Image, display
from pathlib import Path

figures = sorted(Path("outputs/figures").glob("*.png"))
for figure in figures[:6]:
    print(figure.name)
    display(Image(filename=str(figure)))

## 11. Đọc chi tiết các file metrics

Tôi đọc từng file JSON của các thí nghiệm để xem độ chính xác, best validation accuracy, số tham số và lịch sử học tập.

In [ ]:
import json
from pathlib import Path

for name in ["msa_custom_patch", "msa_torch_patch", "tok_patch_custom", "tok_row_custom", "tok_cnn_stem_custom"]:
    p = Path(f"outputs/metrics/{name}.json")
    if p.exists():
        with p.open("r", encoding="utf-8") as f:
            data = json.load(f)
        print(f"\n=== {name} ===")
        print("test_accuracy:", data.get("test", {}).get("accuracy"))
        print("best_validation_accuracy:", data.get("best_validation_accuracy"))
        print("num_parameters:", data.get("num_parameters"))
        print("history_keys:", list(data.get("history", {}).keys()))
    else:
        print(f"\n=== {name} ===")
        print("Chưa có file metrics cho experiment này.")

## 12. Kết luận và nhận xét

Sau khi chạy xong các thí nghiệm, tôi sẽ so sánh hiệu suất của các cấu hình MSA và tokenizer để rút ra nhận xét về mô hình nào thích hợp hơn cho bài toán phân loại ảnh Fashion-MNIST. Các nhận xét định lượng chỉ được viết sau khi có đủ metrics thật từ quá trình chạy.